In [1]:
import json
from contextlib import suppress
from datetime import datetime, timedelta
from pathlib import Path

import boto3
import numpy as np
import s3fs
import xarray as xr
from authlib.integrations.requests_client import OAuth2Session
from sentinelhub import SHConfig
from tqdm import tqdm

/home/jonas/Documents/Projects/2024/change-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
config = SHConfig()
client = OAuth2Session(config.sh_client_id, config.sh_client_secret)
token = client.fetch_token(config.sh_token_url)

In [10]:
process_api = "https://services.sentinel-hub.com/api/v1/process"
data_folder = Path("./data")
data_folder.mkdir(exist_ok=True)


def base_request(data: list, evalscript: str):
    res = 5  # meters

    aoi = [696920, 4528660, 700140, 4532020]
    crs = "http://www.opengis.net/def/crs/EPSG/0/32614"
    return {
        "input": {"bounds": {"bbox": aoi, "properties": {"crs": crs}}, "data": data},
        "output": {
            "resx": res,
            "resy": res,
            "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}],
        },
        "evalscript": evalscript,
    }

In [23]:
with open("beta_2harm.js", "r") as src:
    beta_evalscript = src.read()

beta_data = [
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
    }
]

beta_request = base_request(beta_data, beta_evalscript)

In [24]:
beta = client.post(process_api, json=beta_request)
out_beta = data_folder / "beta_2harm.tif"

beta.raise_for_status()

with open(out_beta, "wb") as f:
    f.write(beta.content)

HTTPError: 400 Client Error: Bad Request for url: https://services.sentinel-hub.com/api/v1/process

In [14]:
# Make new Bucket
region = "eu-central-1"
bucket_name = "sh-change-detection-test"

boto3.setup_default_session(profile_name="default")
s3_client = boto3.client("s3", region_name=region)
location = {"LocationConstraint": region}
with suppress(s3_client.exceptions.BucketAlreadyOwnedByYou):
    s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)

In [15]:
# Set SH BYOC bucket policy
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "Sentinel Hub permissions",
            "Effect": "Allow",
            "Principal": {"AWS": "arn:aws:iam::614251495211:root"},
            "Action": ["s3:GetBucketLocation", "s3:ListBucket", "s3:GetObject"],
            "Resource": [f"arn:aws:s3:::{bucket_name}", f"arn:aws:s3:::{bucket_name}/*"],
        }
    ],
}

# Convert the policy from JSON dict to string
bucket_policy = json.dumps(bucket_policy)

# Set the new policy
s3 = boto3.client("s3")
s3.put_bucket_policy(Bucket=bucket_name, Policy=bucket_policy)

{'ResponseMetadata': {'RequestId': 'J1D90RXVNCWZE48R',
  'HostId': 'waowSXMmBKngxvgmRLSD4GB0t+I5pu5ZQEKzQfbJ0lHsThv/UzDNKAw+qY4lcQLjaCw7Dnw0yeQEUP3weJ7QGg==',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'x-amz-id-2': 'waowSXMmBKngxvgmRLSD4GB0t+I5pu5ZQEKzQfbJ0lHsThv/UzDNKAw+qY4lcQLjaCw7Dnw0yeQEUP3weJ7QGg==',
   'x-amz-request-id': 'J1D90RXVNCWZE48R',
   'date': 'Sat, 20 Apr 2024 13:47:46 GMT',
   'server': 'AmazonS3'},
  'RetryAttempts': 0}}

In [16]:
beta_ds = xr.open_dataset(out_beta, band_as_variable=True)

Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as ExtraSamples.
Warning 1: TIFFReadDirectory:Sum of Photometric type-related color channels and ExtraSamples doesn't match SamplesPerPixel. Defining non-color channels as 

In [41]:
order = ["y", "x"]
reordered_indexes = {index_name: beta_ds.indexes[index_name] for index_name in order}
beta_ds = beta_ds.reindex(reordered_indexes)

In [27]:
# map to rename from default data variable names
coefficients_map = {data_variable: data_variable.replace("band_", "c") for data_variable in beta_ds.keys()}

In [28]:
beta_ds = beta_ds.rename(coefficients_map)
beta_ds["sigma"] = xr.zeros_like(beta_ds["c1"])
beta_ds["process"] = xr.zeros_like(beta_ds["c1"])

In [29]:
zarr_name = "2harm.zarr"
s3_out = s3fs.S3FileSystem(anon=False, profile="default")
store_out = s3fs.S3Map(root=f"s3://{bucket_name}/{zarr_name}", s3=s3_out, check=False)
beta_ds.to_zarr(store_out, mode="a")

In [30]:
# If session token needs to be refreshed:
s3fs.S3FileSystem.clear_instance_cache()

In [31]:
# Make new Zarr storage on SH
zarr_api = "https://services.sentinel-hub.com/api/v1/zarr/collections"
zarr_data = {
    "name": "Monitoring",
    "s3Bucket": bucket_name,
    "path": f"{zarr_name}/",
    "crs": beta_request["input"]["bounds"]["properties"]["crs"],
}
zarr_byoc = client.post(zarr_api, json=zarr_data)
zarr_byoc.raise_for_status()
zarr_id = zarr_byoc.json()["data"]["id"]

In [32]:
zarr_id

'3799bb79-fb5e-4e4b-afa0-3095772b3ba9'

In [58]:
with open("residuals.js", "r") as src:
    sigma_evalscript = src.read()

sigma_data = [
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
        "id": "ARPS",
    },
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": f"zarr-{zarr_id}",
        "id": "beta",
    },
]

sigma_request = base_request(sigma_data, sigma_evalscript)

sigma = client.post(process_api, json=sigma_request)

In [59]:
out_sigma = data_folder / "sigma.tif"

with open(out_sigma, "wb") as f:
    f.write(sigma.content)

In [74]:
sigma_ds = xr.open_dataset(out_sigma, band_as_variable=True)
beta_ds["sigma"] = sigma_ds["band_1"]

In [76]:
# This updates the sigma on cloud storage
beta_ds.to_zarr(store_out, mode="a")

In [99]:
# Updating process values
date = "2022-01-03"
with open("predict_single_date.js", "r") as src:
    monitor_evalscript = src.read()

monitor_evalscript = monitor_evalscript.replace("INSERT_DATE", date)
# Test with updating process value on a single date

monitor_data = [
    {
        "dataFilter": {"timeRange": {"from": f"{date}T00:00:00Z", "to": f"{date}T23:59:59Z"}},
        "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
        "id": "ARPS",
    },
    {
        "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
        "type": f"zarr-{zarr_id}",
        "id": "beta",
    },
]

monitor_request = base_request(monitor_data, monitor_evalscript)
monitor = client.post(process_api, json=monitor_request)
monitor.raise_for_status()

out_monitor = data_folder / f"{date}.tif"

with open(out_monitor, "wb") as f:
    f.write(monitor.content)

In [109]:
def update_process(date):
    # Updating process values
    with open("predict_single_date.js", "r") as src:
        monitor_evalscript = src.read()

    monitor_evalscript = monitor_evalscript.replace("INSERT_DATE", date)
    # Test with updating process value on a single date

    monitor_data = [
        {
            "dataFilter": {"timeRange": {"from": f"{date}T00:00:00Z", "to": f"{date}T23:59:59Z"}},
            "type": "byoc-b690a8ba-05c4-49dc-91c7-8484a1007176",
            "id": "ARPS",
        },
        {
            "dataFilter": {"timeRange": {"from": "2021-01-01T00:00:00Z", "to": "2022-01-01T00:00:00Z"}},
            "type": f"zarr-{zarr_id}",
            "id": "beta",
        },
    ]

    monitor_request = base_request(monitor_data, monitor_evalscript)
    monitor = client.post(process_api, json=monitor_request)
    monitor.raise_for_status()

    out_monitor = data_folder / f"{date}.tif"

    with open(out_monitor, "wb") as f:
        f.write(monitor.content)

    zarr_name = "monitoring.zarr"
    s3_out = s3fs.S3FileSystem(anon=False, profile="default")
    store_out = s3fs.S3Map(root=f"s3://{bucket_name}/{zarr_name}", s3=s3_out, check=False)
    monitor_ds = xr.open_dataset(out_monitor, band_as_variable=True)
    beta_ds["process"] = monitor_ds["band_1"]
    # This updates the sigma on cloud storage
    beta_ds.to_zarr(store_out, mode="a")

In [100]:
d0 = datetime(2022, 1, 1)
d1 = datetime(2022, 2, 1)
dt = timedelta(days=1)
dates = np.arange(d0, d1, dt).astype(datetime)

In [122]:
for date in tqdm(dates):
    update_process(date.strftime("%Y-%m-%d"))

100%|██████████| 31/31 [03:04<00:00,  5.96s/it]


In [113]:
beta_ds["process"] = xr.zeros_like(beta_ds["c1"])